In [11]:
import numpy as np
from numpy.linalg import solve
from scipy.linalg import eig
from dataclasses import dataclass
from typing import Optional, Callable, Tuple
import matplotlib.pyplot as plt
from scipy.special import j1
from types import SimpleNamespace
import warnings
import os
import time
# Trapezoid shim (NumPy renamed trapz->trapezoid)
try:
    from numpy import trapezoid as trapz
except Exception:
    try:
        from scipy.integrate import trapezoid as trapz
    except Exception:
        from numpy import trapz
Complex = np.complex128
from numba import njit, prange
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import math
from typing import List

from tqdm.auto import tqdm  # optional
from numba import set_num_threads, get_num_threads
set_num_threads(8)         # or whatever
print("Numba threads:", get_num_threads())

from scipy.constants import epsilon_0

import utility_files as util

from scipy.constants import epsilon_0

np.set_printoptions(
    precision=3,
    suppress=False,
)


%load_ext autoreload
%autoreload 2

Numba threads: 8
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# ---------------------------------------------------------------------------
# Voigt 6 ↔ 4th-rank converters
# ---------------------------------------------------------------------------

# Standard 6-index Voigt mapping:
# 0↔(0,0), 1↔(1,1), 2↔(2,2), 3↔(1,2), 4↔(0,2), 5↔(0,1)

_VOIGT_PAIRS = [(0,0),(1,1),(2,2),(1,2),(0,2),(0,1)]
_ALPHA = np.array([1,1,1,1,1,1], float)
_VOIGT_6_LOOKUP = {}
for a, (i, j) in enumerate(_VOIGT_PAIRS):
    _VOIGT_6_LOOKUP[(i, j)] = a
    _VOIGT_6_LOOKUP[(j, i)] = a  # minor symmetry

# --- Voigt → full 4th-rank mapping ---

_voigt_to_pair = np.array([
    [0, 0],  # 0 -> 11
    [1, 1],  # 1 -> 22
    [2, 2],  # 2 -> 33
    [1, 2],  # 3 -> 23
    [0, 2],  # 4 -> 13
    [0, 1],  # 5 -> 12
], dtype=np.int64)

KIND_ELASTIC = np.int8(0)
KIND_PIEZO   = np.int8(1)


C6_LN = np.array([
    [203.0, 57.3, 75.2,  -8.5,  0.0,  0.0],
    [ 57.3,203.0, 75.2,   8.5,  0.0,  0.0],
    [ 75.2, 75.2,242.4,   0.0,  0.0,  0.0],
    [ -8.5,  8.5,  0.0,  59.5,  0.0,  0.0],
    [  0.0,  0.0,  0.0,   0.0, 59.5, -8.5],
    [  0.0,  0.0,  0.0,   0.0, -8.5, (203.0-57.3)/2],
], np.float64) * 1e9

rho_LN = 4650.0
L_LN   = 115e-6

#import numpy as np

# LiNbO3 piezoelectric stress coefficients (C/m^2)
e15 = 3.83
e22 = 2.37
e31 = 0.23
e33 = 1.30

# 3×6 Voigt-form piezoelectric tensor [e]
# Rows:    D1,   D2,   D3
# Cols: S1,S2,S3,S4,S5,S6  (11,22,33,23,13,12)
e6_LN = np.array([
    [   0.0,   0.0,   0.0,   0.0,  e15, -e22],
    [ -e22,   e22,   0.0,  e15,   0.0,  0.0],
    [  e31,   e31,   e33,  0.0,   0.0,  0.0]
], dtype=np.float64)

# You fill these with your actual tensors:
#   e6_LN   : (3x6) piezo matrix
#   eps3_LN : (3x3) dielectric tensor
#e6_LN   = ...  # TODO: insert LiNbO3 e_ij in Voigt
#eps3_LN = ...  # TODO: insert LiNbO3 permittivity tensor
eps11 = 44.3
eps33=27.9

eps3_LN = np.array([[44.3,0.0,0.0],
                 [0.0,44.3,0.0],
                 [0.0,0.0,27.9]], dtype=np.float64)*epsilon_0
# after defining e6_LN
e_ijk_LN = util.e6_voigt_to_eijk(e6_LN)
tan_LN=5e-3
tan_eps_LN=0.0

# ---------- Sapphire ----------
C6_sap = np.array([
    [497,164,113,  -23,  0,  0],
    [164,497,113,   23,  0,  0],
    [113,113,501,    0,  0,  0],
    [ -23,  23,  0, 147,  0,  0],
    [  0,   0,  0,   0, 147,-23],
    [  0,   0,  0,   0,-23,(497-164)/2],
],  dtype=np.float64) * 1e9  # Pa

rho_sap = 3980.0   # kg/m^3
L_sap   = 1e-2    # m
tan_sap = 4e-4

# ---------- Indium (isotropic) ----------
B_in   = 35.0e9      # bulk modulus [Pa]
G_in   = 4.4e9       # shear modulus [Pa]
rho_in = 1928.3      # density [kg/m^3]

lam_in = B_in - 2.0*G_in/3.0   # Lamé λ
mu_in  = G_in                  # Lamé μ

C11_in = 192#lam_in + 2.0*mu_in
C12_in =163# lam_in
C44_in =42 # mu_in

C6_in = np.array([
    [C11_in, C12_in, C12_in, 0.0,   0.0,   0.0],
    [C12_in, C11_in, C12_in, 0.0,   0.0,   0.0],
    [C12_in, C12_in, C11_in, 0.0,   0.0,   0.0],
    [0.0,    0.0,    0.0,    C44_in,0.0,   0.0],
    [0.0,    0.0,    0.0,    0.0,   C44_in,0.0],
    [0.0,    0.0,    0.0,    0.0,   0.0,   C44_in],
], dtype=np.float64)*1e9

L_in   = 350e-9   # m
tan_in = 1e-2

f_ref_b=1e8
omega_ref = 2.0 * np.pi * f_ref_b

theta_cut=90-36.7

# frequency grid
df    = 0.05e6
f_min = 10e6
f_max = 1e9
Nf    = int((f_max - f_min)/df) + 1
freqs = np.linspace(f_min, f_max, Nf)

# aperture + slowness integration params
a_radius      = 5e-4
Nphi          = 90

du=np.pi/24
# block params
R_max  = 1.5
tol_im = 1e-15
tol_s  = 1e-18

# BCs
mech_bc_flag_topwall  = 0    # 0 free, 1 clamped
top_elec_flag         = 3    # passthrough at top (TEM handles electrical)
top_Z_elec            = 0.0 + 0.0j
top_passthrough_idx   = 3    # your chosen free amplitude index

mech_bc_flag_backwall = 0    # sapphire back surface free

Z0 = 50.0

In [25]:
C4_LN = util.voigt6_to_cijkl(C6_LN, _VOIGT_PAIRS)
C4_sap = util.voigt6_to_cijkl(C6_sap, _VOIGT_PAIRS)

rot_C4_LN_eff = util.rotate_tensor_4_rank(C4_LN,theta_cut)
rot_e_ijk_LN=util.rotate_tensor_3_rank(e_ijk_LN,theta_cut)
rot_eps3_LN=util.rotate_tensor_2_rank(eps3_LN,theta_cut)

layers= [
    {
        "name": "LiNbO3",
        "kind": "piezo",       # <— tells the cache builder to use piezo kernel
        "C4_eff": rot_C4_LN_eff,   # 3×3×3×3 complex stiffness
        "e_ijk": rot_e_ijk_LN,     # 3×3×3 piezo tensor
        "eps_tensor": rot_eps3_LN,  # 3×3 complex permittivity
        "rho": rho_LN,
        "L": L_LN,
        "tan_delta": tan_LN,
        "tan_eps": tan_eps_LN,

        # Electric BC at the *bottom* interface of the piezo layer
        "bottom_elec_flag": 0,      # 0 short, 1 open, 2 Z
        "bottom_Z_elec": 0.0 + 0.0j,
    },
    # {
    #     "name": "Indium",
    #     "kind": "elastic",
    #     "C4_eff": C4_in_eff,
    #     "rho": rho_in,
    #     "L": L_in,
    #     "tan_delta": tan_in,
    # },
    {
        "name": "Sapphire",
        "kind": "elastic",
        "C4_eff": C4_sap,
        "rho": rho_sap,
        "L": L_sap,
        "tan_delta": tan_sap,
    },
]

td_fns_list = [
    lambda f: util.tan_delta_powerlaw(f, tan0=3e-4, f0_hz=100e6, exponent=0.3, tan_max=2e-3),
    lambda f: util.tan_delta_debye(f, tau_s=1/(2*np.pi*80e6), Delta=2e-3, tan_bg=4e-4, tan_max=5e-3),
]

freqs = np.asarray(freqs, dtype=np.float64)
edges, f_ref_blocks = util.make_log_blocks_from_ratio(f_min, f_max, R_max=R_max)

omegas_upper=[]

min_omega_upper=40
max_omega_upper=400
for i in range(len(f_ref_blocks)):
    if edges[i]<=2e6:
        omeg=max_omega_upper
    elif 2e6<=edges[i]<=10e6:
        omeg=400
    elif 10e6<=edges[i]<=20e6:
        omeg=1250
    elif 20e6<=edges[i]<=50e6:
        omeg=100
    elif 50e6<=edges[i]<=100e6:
        omeg=75
    else:
        omeg=min_omega_upper
    omegas_upper.append(omeg)

omega_s_a_max=np.array(omegas_upper,dtype=np.float64)

In [32]:
td_fns = {
    "LiNbO3":  lambda f: util.tan_delta_powerlaw(f, tan0=4e-4, f0_hz=50e6, exponent=0.7, tan_max=8e-3),
    "Sapphire":lambda f: min(
        util.tan_delta_debye(f, tau=1/(2*np.pi*50e6), Delta=4e-3, tan_bg=1e-4),
        1e-4
    ),
}

blocks = util.build_slowness_blocks_streaming(
    freqs=freqs,
    edges=edges,
    f_ref_blocks=f_ref_blocks,
    omega_s_a_max=omega_s_a_max,
    du=du,
    Nphi=Nphi,
    a_radius=a_radius,
    layers=layers,

    mech_bc_flag_topwall=mech_bc_flag_topwall,
    top_elec_flag=top_elec_flag,
    top_Z_elec=top_Z_elec,
    top_elec_passthrough_idx=3,
    mech_bc_flag_backwall=mech_bc_flag_backwall,

    tan_delta_block_fns_per_layer=td_fns,   # ✅ now accepted
    store_top_full_modes=False,
    store_top_Bf=True,
    KIND_ELASTIC=KIND_ELASTIC,
    KIND_PIEZO=KIND_PIEZO
)

print("Number of blocks:", len(blocks))

# 3) Inspect the first block to confirm layout
blk0 = blocks[0]
out0 = blk0[7]  # (idx, omegas, s_vals, const_prefac, piston, L_layers, n_modes, out)

print("len(out0) =", len(out0))
print("out0 shapes:", [getattr(x, "shape", None) for x in out0])

# Expected:
# len(out0) == 14
# out0[8].shape  == (Ns_b, Nphi, 4, 4)   # B_top_arr
# out0[9].shape  == (Ns_b, Nphi, 4)      # f_top_arr
# out0[10:14]    == (Ns_b, Nphi, 4) each # phi+/phi-/q+/q-



Building slowness blocks:   0%|          | 0/12 [00:00<?, ?it/s]

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
[1m[1mUntyped global name 'fill_layer_modes_kernel':[0m [1m[1mCannot determine Numba type of <class 'function'>[0m
[1m
File "utility_files.py", line 1212:[0m
[1mdef build_block_caches_streaming_fill_kernel(
    <source elided>
    Nphi     = cos_phi.shape[0]
[1m    N_layers = kind.shape[0]
[0m    [1m^[0m[0m
[0m
[0m[1mDuring: Pass nopython_type_inference[0m

In [ ]:
# ----------------------------------------
blocks_elec = compress_blocks_fulltop_to_elec_only(
    blocks,
    mech_bc_flag_topwall=mech_bc_flag_topwall,
    top_passthrough_idx=3,
)

# ----------------------------------------
# 3) Integrate spectrum (your in-place version)
# ----------------------------------------
# L_layers is just the layer thickness array in order:



In [ ]:
L_layers = np.array([lay["L"] for lay in layers], dtype=np.float64)

In [ ]:
I_omega2 = integrate_all_blocks_streaming_inplace_elec_only(
    freqs=freqs,
    blocks=blocks_elec,
    L_layers=L_layers,
    a_radius=a_radius,
    Z0=Z0,
    show_progress_blocks=True,
)

In [ ]:
Zimp2=50*(1+I_omega2)/(1-I_omega2)

In [ ]:
ZimpE=np.conj(Zimp2)

In [ ]:
print("I_omega[0], I_omega[-1] =", I_omega[0], I_omega[-1])



In [ ]:
%matplotlib widget
#freqplt=np.where(freqs>=10e6 and freqs<=200e6)
      # optional: nuke old widget figures
#fmin, fmax = 0e6, 200e6
m = (freqs >= f_min) & (freqs <= f_max)

fig, ax = plt.subplots()
#ax.plot(freqs[m]/1e6, np.imag((Ze[m])),label="Re(s(t)) Mason", color="orange")
ax.plot(freqs[m]/1e6, np.imag(ZimpE[m]),label="Re(s(t)) Mason", color="blue")

#ax.plot(freqs[m]/1e6, np.imag(np.conj(Zimp3[m])))
#ax.plot(freqs[m]/1e6, np.imag(np.conj(Zimp4[m])))
#ax.set_xlim(fmin/1e6, fmax/1e6)
ax.set_xlabel("Frequency (MHz)")
ax.set_ylabel(r"Im$(Z)$ ($\Omega$)")
ax.set_title(r"Imaginary Impedance: 0.05–500 MHz")  # title
ax.legend()
plt.ylim(-2000,500)
fig.tight_layout()
fig.savefig("Zimag_to500MHz_zoom2.png", dpi=300, bbox_inches="tight")
plt.show()




In [ ]:
%matplotlib widget
#freqplt=np.where(freqs>=10e6 and freqs<=200e6)
      # optional: nuke old widget figures
#fmin, fmax = 0e6, 200e6
m = (freqs >= f_min) & (freqs <= f_max)

fig, ax = plt.subplots()
#ax.plot(freqs[m]/1e6, np.real((Ze[m])),label="Re(s(t)) Mason", color="orange")
ax.plot(freqs[m]/1e6, np.real(np.conj(Zimp2[m])),label="Re(s(t)) Mason", color="blue")

#ax.plot(freqs[m]/1e6, np.imag(np.conj(Zimp3[m])))
#ax.plot(freqs[m]/1e6, np.imag(np.conj(Zimp4[m])))
#ax.set_xlim(fmin/1e6, fmax/1e6)
ax.set_xlabel("Frequency (MHz)")
ax.set_ylabel(r"Im$(Z)$ ($\Omega$)")
ax.set_title(r"Imaginary Impedance: 0.05–500 MHz")  # title
ax.legend()
#plt.ylim(-2000,500)
fig.tight_layout()
fig.savefig("Zreal_to500MHz.png", dpi=300, bbox_inches="tight")
plt.show()



In [ ]:
#%matplotlib widget
#freqplt=np.where(freqs>=10e6 and freqs<=200e6)
      # optional: nuke old widget figures
      # optional: nuke old widget figures
#fmin, fmax = 0e6, 300e6
m = (freqs >= f_min) & (freqs <= f_max)

fig, ax = plt.subplots()
ax.plot(freqs[m]/1e6, np.abs(np.conj(Zimp2[m])))
# ax.plot(freqs[m]/1e6, np.abs(np.conj(Zimp2[m])))
# ax.plot(freqs[m]/1e6, np.abs(np.conj(Zimp3[m])))
# ax.plot(freqs[m]/1e6, np.abs(np.conj(Zimp4[m])))
#ax.set_xlim(fmin/1e6, fmax/1e6)
ax.set_xlabel("Frequency (MHz)")
ax.set_ylabel(r"$|Z|$ ($\Omega$)")
ax.set_title(r"Absolute Impedance: 10–200 MHz")  # title

fig.tight_layout()
#fig.savefig("Zabsolute_200to400MHz.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
plt.close('all')

In [ ]:
# covered = np.zeros(len(freqs), dtype=np.int8)
# for blk in blocks_elec:
#     idx_b = blk[0]
#     covered[idx_b] = 1
# print("uncovered:", np.where(covered == 0)[0])

In [ ]:
from typing import Any, Dict, List, Optional, Tuple

def _as_scalar_C(x: Any, axis: int = 2) -> complex:
    """Accept scalar or 3x3x3x3 and return C_zzzz."""
    if np.isscalar(x):
        return complex(x)
    a = np.asarray(x)
    if a.ndim == 4:
        return complex(a[axis, axis, axis, axis])
    raise ValueError(f"Unsupported C input shape {a.shape}; use scalar or 3x3x3x3.")

def _as_scalar_e(x: Any, axis: int = 2) -> complex:
    """Accept scalar or 3x3x3 and return e_zzz (i.e. e33 in thickness-only model)."""
    if np.isscalar(x):
        return complex(x)
    a = np.asarray(x)
    if a.ndim == 3:
        return complex(a[axis, axis, axis])
    raise ValueError(f"Unsupported e input shape {a.shape}; use scalar or 3x3x3.")

def _as_scalar_eps(x: Any, axis: int = 2) -> complex:
    """Accept scalar or 3x3 and return eps_zz."""
    if np.isscalar(x):
        return complex(x)
    a = np.asarray(x)
    if a.ndim == 2:
        return complex(a[axis, axis])
    raise ValueError(f"Unsupported eps input shape {a.shape}; use scalar or 3x3.")

def build_kwargs_for_analytic_impedance(
    layers: List[Dict[str, Any]],
    *,
    A: float,
    axis: int = 2,
    i_sign: int = -1,
    # how to treat loss:
    #   "embedded": constants already complex -> pass tan_* = 0
    #   "tandelta": use real baselines + pass tan_delta/tan_eps
    loss_mode: str = "embedded",
    # sapphire fix: conjugate stiffness for these layer names (case-insensitive)
    conj_names: Tuple[str, ...] = ("Sapphire",),
    # optionally override N (otherwise uses your N = C0*h33)
    N_override: Optional[complex] = None,
) -> Dict[str, Any]:
    """
    Returns kwargs for:
        Za, Ze = electrical_impedance_analytic_stack(freqs, **kwargs)

    Expects layers in order: [piezo, elastic1, elastic2] (or at least: one piezo + two elastics).
    Accepts either tensor-valued or scalar-valued C/e/eps fields.

    Piezo:
      cE = C_zzzz, e33 = e_zzz, eps33 = eps_zz
      cD = cE + e33^2/eps33
      v0 = sqrt(cD/rho0)

    Elastics:
      c = C_zzzz
      v = sqrt(c/rho)
      if name in conj_names: c <- conj(c)   (fix "-ω" convention)

    N:
      C0 = eps33*A/d
      h33 = e33/eps33
      N = C0*h33 (= e33*A/d)
    """
    piezo = next(L for L in layers if L.get("kind", "").lower() == "piezo")
    elastics = [L for L in layers if L.get("kind", "").lower() == "elastic"]
    if len(elastics) < 2:
        raise ValueError("Need at least two elastic layers for this 3-layer analytic model.")
    L1, L2 = elastics[0], elastics[1]

    # ---- PIEZO scalars ----
    cE = _as_scalar_C(piezo["C4_eff"], axis=axis)
    e33 = _as_scalar_e(piezo["e_ijk"], axis=axis)
    eps33 = _as_scalar_eps(piezo["eps_tensor"], axis=axis)

    rho0 = float(piezo["rho"])
    d = float(piezo["L"])

    # Loss handling
    if loss_mode.lower() == "tandelta":
        cE_use = float(np.real(cE))
        eps_use = float(np.real(eps33))
        e_use = complex(e33)  # usually real; keep complex if you store it that way
        tan_delta0 = float(piezo.get("tan_delta", 0.0))
        tan_eps = float(piezo.get("tan_eps", 0.0))
        C0_real = eps_use * A / d
    elif loss_mode.lower() == "embedded":
        cE_use = cE
        eps_use = eps33
        e_use = e33
        tan_delta0 = 0.0
        tan_eps = 0.0
        C0_real = eps_use * A / d  # may be complex; tan_eps=0 so not re-applied
    else:
        raise ValueError("loss_mode must be 'embedded' or 'tandelta'.")

    cD_use = cE_use + (e_use**2) / eps_use
    v0 = np.sqrt(cD_use / rho0)

    h33 = e_use / eps_use
    N = (C0_real * h33) if (N_override is None) else N_override

    # ---- ELASTIC 1 scalars ----
    c1 = _as_scalar_C(L1["C4_eff"], axis=axis)
    if loss_mode.lower() == "tandelta":
        c1_use = float(np.real(c1))
        tan_delta1 = float(L1.get("tan_delta", 0.0))
    else:
        c1_use = c1
        tan_delta1 = 0.0
    rho1 = float(L1["rho"])
    l1 = float(L1["L"])
    v1 = np.sqrt(c1_use / rho1)

    # ---- ELASTIC 2 scalars (Sapphire conjugation if needed) ----
    c2 = _as_scalar_C(L2["C4_eff"], axis=axis)
    if any(L2.get("name","").lower() == nm.lower() for nm in conj_names):
        c2 = np.conjugate(c2)

    if loss_mode.lower() == "tandelta":
        c2_use = float(np.real(c2))
        tan_delta2 = float(L2.get("tan_delta", 0.0))
    else:
        c2_use = c2
        tan_delta2 = 0.0

    rho2 = float(L2["rho"])
    l2 = float(L2["L"])
    v2 = np.sqrt(c2_use / rho2)

    return dict(
        C0_real=C0_real,
        N=N,
        rho0=rho0, v0=v0, d=d,
        rho1=rho1, v1=v1, l1=l1,
        rho2=rho2, v2=v2, l2=l2,
        tan_delta0=tan_delta0,
        tan_delta1=tan_delta1,
        tan_delta2=tan_delta2,
        tan_eps=tan_eps,
        A=A,
        i_sign=i_sign,
    )

In [ ]:


def _apply_mech_loss_to_velocity(v: complex | float, tan_delta: float) -> complex:
    """
    Mechanical loss tangent via complex modulus:
        M* = M (1 + i tan_delta)  =>  v* = v * sqrt(1 + i tan_delta)
    """
    return v * np.sqrt(1.0 + 1j * tan_delta)



In [ ]:

def acoustic_impedance_analytic(
    freq_hz: float | np.ndarray,
    # REQUIRED (positional) parameters
    rho0: float, v0: float | complex, d: float,
    rho1: float, v1: float | complex, l1: float,
    rho2: float, v2: float | complex, l2: float,
    *,
    # OPTIONAL (keyword-only) parameters
    tan_delta0: float = 0.0,
    tan_delta1: float = 0.0,
    tan_delta2: float = 0.0,
    A: float = 1.0,
    i_sign: int = -1,
) -> np.ndarray:
    """
    Your closed-form acoustic impedance:

        Za = (-i Z0 / (1 - cos(k0 d))) *
             ( Z1*tan(k1 l1 + phi)*cos(k0 d) + Z0*sin(k0 d) ) /
             ( Z1*tan(k1 l1 + phi)*sin(k0 d) + 2 Z0 )

        phi = arctan( (Z2/Z1) * tan(k2 l2) )
        k = omega / v,  Z = rho*v*A

    Adds mechanical loss via tan_delta0/1/2 (implemented by complex velocity).
    """
    f = np.asarray(freq_hz, dtype=float)
    omega = 2.0 * np.pi * f

    # Apply mechanical losses
    v0c = _apply_mech_loss_to_velocity(v0, tan_delta0)
    v1c = _apply_mech_loss_to_velocity(v1, tan_delta1)
    v2c = _apply_mech_loss_to_velocity(v2, tan_delta2)

    # Wavenumbers
    k0 = omega / v0c
    k1 = omega / v1c
    k2 = omega / v2c

    # Characteristic impedances (force/velocity)
    Z0 = rho0 * v0c * A
    Z1 = rho1 * v1c * A
    Z2 = rho2 * v2c * A

    # Phase shift from layer 2
    phi = np.arctan((Z2 / Z1) * np.tan(k2 * l2))

    cos0 = np.cos(k0 * d)
    sin0 = np.sin(k0 * d)

    T = Z1 * np.tan(k1 * l1 + phi)

    pref = (i_sign * 1j) * Z0 / (1.0 - cos0)
    num  = T * cos0 + Z0 * sin0
    den  = T * sin0 + 2.0 * Z0

    return pref * (num / den)



In [ ]:

def electrical_impedance_from_Za(
    freq_hz: float | np.ndarray,
    C0_real: float | complex,
    N: complex,
    Za: complex | np.ndarray,
    *,
    tan_eps: float = 0.0,
) -> np.ndarray:
    """
    Your electrical impedance formula, with dielectric loss tangent tan_eps applied to C0:

        C0* = C0_real * (1 - i tan_eps)

        Z_elec(ω) = 1/(i ω C0*) * ( (Za - N^2/(i ω C0*)) / Za )
    """
    f = np.asarray(freq_hz, dtype=float)
    omega = 2.0 * np.pi * f
    Za = np.asarray(Za, dtype=np.complex128)

    C0c = C0_real * (1.0 - 1j * tan_eps)
    iwC0 = 1j * omega * C0c

    Ze = (1.0 / iwC0) * ((Za - (N**2) / iwC0) / Za)
    Ze = np.where(Za == 0, np.inf + 0j, Ze)
    return Ze


In [ ]:

def electrical_impedance_analytic_stack(
    freq_hz: float | np.ndarray,
    C0_real: float | complex,
    N: complex,
    rho0: float, v0: float | complex, d: float,
    rho1: float, v1: float | complex, l1: float,
    rho2: float, v2: float | complex, l2: float,
    *,
    tan_delta0: float = 0.0,
    tan_delta1: float = 0.0,
    tan_delta2: float = 0.0,
    tan_eps: float = 0.0,
    A: float = 1.0,
    i_sign: int = -1,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Convenience wrapper: returns (Za, Z_elec).
    """
    Za = acoustic_impedance_analytic(
        freq_hz,
        rho0, v0, d,
        rho1, v1, l1,
        rho2, v2, l2,
        tan_delta0=tan_delta0,
        tan_delta1=tan_delta1,
        tan_delta2=tan_delta2,
        A=A,
        i_sign=i_sign,
    )
    Ze = electrical_impedance_from_Za(
        freq_hz, C0_real=C0_real, N=N, Za=Za, tan_eps=tan_eps
    )
    return Za, Ze

In [ ]:

# plt.figure()
# plt.plot(freqs*1e-6, np.abs(Ze), label="|Z_elec|")
# plt.xlabel("Frequency (MHz)")
# plt.ylabel("Magnitude (Ohm)")
# plt.title("Magnitude of electrical impedance")
# plt.legend()
# #plt.tight_layout()
# plt.show()

In [ ]:
# %matplotlib widget
# #freqplt=np.where(freqs>=10e6 and freqs<=200e6)
#       # optional: nuke old widget figures
# fmin, fmax = 1e6, 300e6
# m = (freqs >= f_min) & (freqs <= f_max)

# fig, ax = plt.subplots()
# ax.plot(freqs[m]/1e6, np.imag(np.conj(Zimp[m])))
# plt.plot(freqs[m]*1e-6, np.imag(Ze[m]), label="Im(Z_elec)")
# #ax.set_xlim(fmin/1e6, fmax/1e6)
# ax.set_xlabel("Frequency (MHz)")
# ax.set_ylabel(r"Im$(Z)$ ($\Omega$)")
# ax.set_title(r"Imaginary Impedance: 10–200 MHz")  # title

# fig.tight_layout()
# #fig.savefig("Zimag_10to200MHz.png", dpi=300, bbox_inches="tight")
# plt.show()


In [ ]:
# %matplotlib widget
# #freqplt=np.where(freqs>=10e6 and freqs<=200e6)
#       # optional: nuke old widget figures
# fmin, fmax = 1e6, 300e6
# m = (freqs >= f_min) & (freqs <= f_max)

# fig, ax = plt.subplots()
# ax.plot(freqs[m]/1e6, np.abs(np.conj(Zimp[m])))
# plt.plot(freqs[m]*1e-6, np.abs(Ze[m]), label="Im(Z_elec)")
# #ax.set_xlim(fmin/1e6, fmax/1e6)
# ax.set_xlabel("Frequency (MHz)")
# ax.set_ylabel(r"Im$(Z)$ ($\Omega$)")
# ax.set_title(r"Imaginary Impedance: 10–200 MHz")  # title

# fig.tight_layout()
# #fig.savefig("Zimag_10to200MHz.png", dpi=300, bbox_inches="tight")
# plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------- helpers ----------
def reflection_coeff(Zin, Z0=50.0):
    Zin = np.asarray(Zin, dtype=np.complex128)
    return (Zin - Z0) / (Zin + Z0)

def window_1d(name, n):
    name = (name or "none").lower()
    if name in ("none", "rect", "rectangular"):
        return np.ones(n)
    if name in ("hann", "hanning"):
        return np.hanning(n)
    if name == "hamming":
        return np.hamming(n)
    if name == "blackman":
        return np.blackman(n)
    raise ValueError(f"Unknown window '{name}'")

def gaussian_spectrum_from_time_fwhm(freqs_hz, f0_hz, t_fwhm_s, t0_s=0.0, amp=1.0):
    f = np.asarray(freqs_hz, dtype=float)
    sigma_t = t_fwhm_s / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    sigma_f = 1.0 / (2.0 * np.pi * sigma_t)
    return amp * np.exp(-0.5 * ((f - f0_hz) / sigma_f) ** 2) * np.exp(-1j * 2.0 * np.pi * f * t0_s)
def rect_pulse_spectrum(
    freqs_hz: np.ndarray,
    *,
    T_s: float,              # pulse duration (seconds)
    t0_s: float = 0.0,       # time shift
    f0_hz: float = 0.0,      # carrier (Hz). f0=0 => baseband rectangle
    amp: float = 1.0,
    make_real: bool = True,# match the IFFT make_real choice
) -> np.ndarray:
    """
    Returns P(f) on f>=0 grid.

    Baseband rectangle (f0=0):
      P(f) = amp * T * sinc(f T) * exp(-i 2π f t0)

    RF burst (f0>0):
      - if make_real=True (real time waveform), include both sidebands:
          P(f) = amp * (T/2) * [sinc((f-f0)T) + sinc((f+f0)T)] * exp(-i 2π f t0)
      - if make_real=False (analytic complex waveform), keep only +f0 lobe:
          P(f) = amp * T * sinc((f-f0)T) * exp(-i 2π f t0)

    numpy.sinc(x) = sin(pi x)/(pi x).
    """
    f = np.asarray(freqs_hz, dtype=float)
    phase = np.exp(-1j * 2.0 * np.pi * f * t0_s)
    phase2=np.exp(1j*f0_hz*2*np.pi*t0_s)
    if make_real:
        return amp * (T_s /(2.0)) * (np.sinc((f0_hz-f) * T_s)*phase2- np.sinc((f + f0_hz) * T_s)*np.conj(phase2)) * phase*(-1j)
    else:
        return amp * T_s * np.sinc((f - f0_hz) * T_s) * phase*(-1j)
def pad_to_dc_grid(freqs_hz, H_pos, fill_value=1.0+0.0j, df=None, rtol_align=1e-8):
    """
    Embed one-sided spectrum H_pos sampled on freqs_hz (uniform, starting at f_min>0)
    into a uniform grid starting at 0 with the same df, filling missing bins with fill_value.
    """
    f = np.asarray(freqs_hz, dtype=float)
    H = np.asarray(H_pos, dtype=np.complex128)

    if df is None:
        df = (f[-1] - f[0]) / (f.size - 1)

    k0 = int(np.rint(f[0] / df))
    kN = int(np.rint(f[-1] / df))

    if not (np.isclose(k0 * df, f[0], rtol=rtol_align, atol=0.0) and
            np.isclose(kN * df, f[-1], rtol=rtol_align, atol=0.0)):
        raise ValueError("Frequency axis is not aligned to a constant df grid.")

    n_full = kN + 1
    f_full = df * np.arange(n_full, dtype=float)
    H_full = np.full(n_full, fill_value, dtype=np.complex128)
    H_full[k0:k0 + H.size] = H
    return f_full, H_full


import numpy as np

def _cosine_taper_onesided(f, *, f_lo=None, f_hi=None, frac=0.10):
    """
    Smooth bandpass/taper for a one-sided f>=0 grid.

    - If f_lo is not None: ramps from 0->1 over a width = frac*(f_hi-f_lo) starting at f_lo.
    - If f_hi is not None: ramps from 1->0 over the same width ending at f_hi.
    - Between ramps: 1.

    This is a Tukey-like cosine taper (flat middle, cosine edges).

    Parameters
    ----------
    f : (N,) array, Hz, assumed increasing
    f_lo : float or None
        Start frequency where passband begins (below this, weight=0 if f_lo is set).
    f_hi : float or None
        End frequency where passband ends (above this, weight=0 if f_hi is set).
    frac : float
        Fraction of the band (f_hi - f_lo) used for EACH taper edge. Typical 0.05–0.2.

    Returns
    -------
    w : (N,) array in [0,1]
    """
    f = np.asarray(f, float)
    if f_lo is None and f_hi is None:
        return np.ones_like(f)

    # Sensible defaults
    if f_lo is None:
        f_lo = float(f[0])
    if f_hi is None:
        f_hi = float(f[-1])

    if not (f_hi > f_lo):
        raise ValueError("Require f_hi > f_lo for taper.")

    bw = f_hi - f_lo
    tw = max(frac * bw, 0.0)

    w = np.zeros_like(f)

    # Flat region
    flat = (f >= f_lo + tw) & (f <= f_hi - tw)
    w[flat] = 1.0

    # Low-end ramp 0 -> 1
    lo = (f >= f_lo) & (f < f_lo + tw) if tw > 0 else (f >= f_lo)
    if np.any(lo) and tw > 0:
        x = (f[lo] - f_lo) / tw  # 0..1
        w[lo] = 0.5 * (1.0 - np.cos(np.pi * x))
    elif np.any(lo) and tw == 0:
        w[lo] = 1.0

    # High-end ramp 1 -> 0
    hi = (f > f_hi - tw) & (f <= f_hi) if tw > 0 else (f <= f_hi)
    if np.any(hi) and tw > 0:
        x = (f_hi - f[hi]) / tw  # 0..1
        w[hi] = np.minimum(w[hi], 0.5 * (1.0 - np.cos(np.pi * x))) if np.any(flat) else 0.5 * (1.0 - np.cos(np.pi * x))
        # simpler: overwrite for hi region (safe because hi doesn't overlap flat except boundary)
        w[hi] = 0.5 * (1.0 - np.cos(np.pi * x))
    elif np.any(hi) and tw == 0:
        # keep whatever was set by lo/flat
        pass

    # Ensure outside [f_lo,f_hi] is zero
    w[(f < f_lo) | (f > f_hi)] = 0.0
    return w


def ifft_one_sided_to_time(
    freqs_hz,
    H_pos,
    *,
    pad_factor: int = 8,
    make_real: bool = True,
    scale_physical: bool = True,
    analytic_preserve_amplitude: bool = True,
    # --- NEW: frequency-domain windowing ---
    window: str | None = "taper",      # None/"taper"
    f_lo: float | None = None,         # e.g. your reliable low cutoff (Hz)
    f_hi: float | None = None,         # e.g. your reliable high cutoff (Hz)
    taper_frac: float = 0.10,          # 5–20% typical
):
    """
    IFFT a one-sided spectrum H_pos defined on a uniform f>=0 grid.

    Windowing:
      If window="taper", applies a smooth cosine taper in frequency to suppress
      hard band edges / DC padding discontinuities that cause ringing/bricks.

      - f_lo: start of passband (below: taper to 0)
      - f_hi: end of passband (above: taper to 0)
      - taper_frac: fraction of (f_hi-f_lo) used for each edge taper
    """
    f = np.asarray(freqs_hz, dtype=float)
    H = np.asarray(H_pos, dtype=np.complex128)

    if f.ndim != 1 or H.ndim != 1 or f.size != H.size:
        raise ValueError("freqs_hz and H_pos must be 1D arrays of equal length.")

    df = np.median(np.diff(f))
    if not np.allclose(np.diff(f), df, rtol=1e-6, atol=0.0):
        raise ValueError("freqs_hz must be uniformly spaced for IFFT.")

    # ---- Apply frequency window BEFORE padding ----
    if window is None:
        Hw = H
    elif window == "taper":
        # Default f_lo/f_hi to the provided grid limits if not specified
        f_lo_eff = f_lo if f_lo is not None else float(f[0])
        f_hi_eff = f_hi if f_hi is not None else float(f[-1])
        w = _cosine_taper_onesided(f, f_lo=f_lo_eff, f_hi=f_hi_eff, frac=taper_frac)
        Hw = H * w
    else:
        raise ValueError(f"Unknown window={window!r}. Use None or 'taper'.")

    n = Hw.size
    n_pad = max(n, int(pad_factor * n))

    Hp = np.zeros(n_pad, dtype=np.complex128)
    Hp[:n] = Hw

    # Optional amplitude preservation for analytic (one-sided) IFFT
    if (not make_real) and analytic_preserve_amplitude:
        # double all positive-frequency bins except DC (index 0)
        Hp[1:n] *= 2.0

    if make_real:
        # Construct two-sided Hermitian spectrum:
        # [0..fmax] + conjugate of [fmax-df .. df]
        Hfull = np.concatenate([Hp, np.conjugate(Hp[-2:0:-1])])
        Nfft = Hfull.size
        ht = np.fft.ifft(Hfull)
    else:
        Nfft = n_pad
        ht = np.fft.ifft(Hp, n=Nfft)

    if scale_physical:
        ht *= (Nfft * df)

    dt = 1.0 / (Nfft * df)
    t = np.arange(Nfft) * dt
    return t, ht

# Pad S down to DC on a uniform grid (zeros below f_min incl DC)


# ============================================================
# 6) Example usage (plot)
# ============================================================




In [ ]:
# import numpy as np

# def ifft_from_onesided(freqs_hz, H_pos, *, scale_physical=True):
#     f = np.asarray(freqs_hz, float)
#     H = np.asarray(H_pos, np.complex128)

#     df = float(np.median(np.diff(f)))
#     if not np.allclose(np.diff(f), df, rtol=1e-6, atol=0):
#         raise ValueError("freqs_hz must be uniformly spaced for a plain IFFT. Resample first.")

#     # Infer full real-FFT length corresponding to this one-sided array
#     # For irfft: one-sided length Npos corresponds to Nfull = 2*(Npos-1)
#     Npos = H.size
#     Nfull = 2 * (Npos - 1)

#     x = np.fft.irfft(H, n=Nfull)
#     t = np.arange(Nfull) / (Nfull * df)  # seconds

#     if scale_physical:
#         # Continuous convention: x(t) ≈ ∫ H(f) e^{i 2π f t} df
#         # Discrete with numpy's 1/N: multiply by N*df
#         x = x * (Nfull * df)

#     return t, x


In [ ]:
# layers2= [
#     {
#         "name": "LiNbO3",
#         "kind": "piezo",       # <— tells the cache builder to use piezo kernel
#         "C4_eff": rot_C4_LN_eff,   # 3×3×3×3 complex stiffness
#         "e_ijk": rot_e_ijk_LN,     # 3×3×3 piezo tensor
#         "eps_tensor": rot_eps3_LN,  # 3×3 complex permittivity
#         "rho": rho_LN,
#         "L": L_LN,
#         "tan_delta": tan_LN,
#         "tan_eps": tan_eps_LN,

#         # Electric BC at the *bottom* interface of the piezo layer
#         "bottom_elec_flag": 0,      # 0 short, 1 open, 2 Z
#         "bottom_Z_elec": 0.0 + 0.0j,
#     },
#     # {
#     #     "name": "Tin",
#     #     "kind": "elastic",
#     #     "C4_eff": C4_in_eff,
#     #     "rho": rho_in,
#     #     "L": 5000e-9,
#     #     "tan_delta": tan_in,
#     # },
#     {
#         "name": "Sapphire",
#         "kind": "elastic",
#         "C4_eff": C4_sap_eff,
#         "rho": rho_sap,
#         "L": L_sap,
#         "tan_delta": tan_sap,
#     },
# ]

In [ ]:
# A = np.pi*(5e-4)**2
# #freqs = np.linspace(0.1e6, 0.9e6, 4000)

# # If your C and eps already contain loss (complex):
# kwargs = build_kwargs_for_analytic_impedance(layers, A=A, loss_mode="tandelta", i_sign=-1)

# # If you want to use tan_delta/tan_eps as the ONLY loss knobs:
# # kwargs = build_kwargs_for_analytic_impedance(layers, A=A, loss_mode="tandelta", i_sign=-1)

# Za, Ze = electrical_impedance_analytic_stack(freqs, **kwargs)



In [ ]:

# ---------- build reflection spectrum ----------
#Gamma = reflection_coeff(Ze, Z0=50.0)
#Gamma *= window_1d('blackman', len(freqs))
# ---------- build reflection spectrum ----------
Gamma2 = reflection_coeff(ZimpE, Z0=50.0)
Gamma2 *= window_1d("blackman", len(freqs))

P = rect_pulse_spectrum(freqs_hz=freqs,T_s=0.5e-6, t0_s=0.25e-6,f0_hz=350e6,amp=1,make_real=True)
# ---------- build echo spectrum ----------
#S = Gamma * P
S2 = Gamma2 * P


In [ ]:
#f_full, S_full = pad_to_dc_grid(freqs, S, fill_value=0.0+0.0j)
f_full2, S_full2 = pad_to_dc_grid(freqs, S2, fill_value=0.0+0.0j)

In [ ]:
plt.close("all")

In [ ]:
##### 

# IFFT to time
t2, s_t2 = ifft_one_sided_to_time(f_full2, S_full2, pad_factor=8, make_real=True,scale_physical=True,analytic_preserve_amplitude= True,window='taper',f_lo=min(freqs), f_hi=max(freqs), taper_frac=0.05)
# IFFT to time
#t, s_t = ifft_one_sided_to_time(f_full2, S_full2, scale_physical=True)
#t3, s_t3 = ifft_one_sided_to_time(f_full, S_full, pad_factor=8, make_real=True)
# ---------- plots ----------
#plt.figure()

#

fig, ax = plt.subplots()


ax.plot(t2*1e6, np.real(s_t2), label="Re(s(t))",       color="blue")
#ax.plot(t*1e6,  np.real(s_t),  label="Re(s(t)) Mason", color="orange")

#ax.plot(t3*1e6,  np.real(s_t3),  label="Re(s(t)) Mason", color="green")
ax.set_xlabel("Time (µs)")
ax.set_ylabel("Amplitude (arb.)")
ax.set_title("Echo train from Γ(f) × Gaussian pulse (DC-grid padded)")
ax.set_xlim(0, 20)
#ax.set_ylim(-32e-6, 32e-6)

ax.legend()
#fig.tight_layout()
#fig.savefig("echo_train.png", dpi=300, bbox_inches="tight")
plt.show()



In [ ]:
plt.figure()
plt.plot(t*1e6, np.abs(s_t2), label="|s(t)|")
plt.xlabel("Time (µs)")
plt.ylabel("Envelope (arb.)")
plt.title("Echo envelope")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.close('all')

In [ ]:
# Cell — load the file
data = np.load("lvm_channels.npz")

# list what’s inside
sorted(data.files)[:10]


In [ ]:
t = data["echo0__t_s"]
amp = data["echo0__amp"]
plt.plot(t*1e6, amp)
plt.xlabel("Time (µs)")
plt.ylabel("Amplitude (arb.)")
plt.title("echo0")
plt.tight_layout()
plt.show()


In [ ]:
# Cell 1 — choose the time gate (e.g. 0 to 1 µs) from echo0


# # Gate in microseconds:
# t0_us, t1_us = 0.0, 1.0
# m = (t >= t0_us*1e-6) & (t <= t1_us*1e-6)

# t_gate = t[m]
# x_gate = amp[m]

# # Basic sanity checks
# dt = np.median(np.diff(t_gate))
# fs = 1.0/dt
# print(f"Gate samples: {len(x_gate)}  |  dt = {dt:.3e} s  |  fs = {fs/1e6:.2f} MHz")


In [ ]:
# Cell 1 — build a pulse record p(t) that lives on the *same indices* as echo0

import numpy as np
import matplotlib.pyplot as plt

def tukey_like(N: int, alpha: float = 0.3) -> np.ndarray:
    """Simple Tukey-style window without SciPy."""
    if N <= 1:
        return np.ones(N)
    alpha = float(alpha)
    if alpha <= 0:
        return np.ones(N)
    if alpha >= 1:
        return np.hanning(N)

    w = np.ones(N)
    edge = int(np.floor(alpha * (N - 1) / 2))
    if edge > 0:
        ramp = 0.5 * (1 - np.cos(np.pi * np.arange(edge) / edge))
        w[:edge] = ramp
        w[-edge:] = ramp[::-1]
    return w

def build_pulse_record_from_echo0(t_s, x, t0_us=0.0, t1_us=1.0, window="tukey", alpha=0.3, remove_mean=True):
    """
    Returns:
      p: pulse-only record, same length as x, aligned on exact sample indices
      idx0, idx1: gate indices in the original record
      dt: sample spacing
    """
    t = t_s - t_s[0]  # relative time
    dt = float(np.median(np.diff(t_s)))  # use full trace spacing

    t0 = t0_us * 1e-6
    t1 = t1_us * 1e-6
    m = (t >= t0) & (t <= t1)
    idx = np.flatnonzero(m)
    if idx.size == 0:
        raise ValueError("Gate produced zero samples. Check t0_us/t1_us.")

    idx0, idx1 = int(idx[0]), int(idx[-1])
    x_gate = x[idx0:idx1+1].copy()

    if remove_mean:
        x_gate = x_gate - np.mean(x_gate)

    if window == "tukey":
        w = tukey_like(len(x_gate), alpha=alpha)
    elif window == "hann":
        w = np.hanning(len(x_gate))
    elif window in (None, "none"):
        w = np.ones(len(x_gate))
    else:
        raise ValueError("window must be 'tukey', 'hann', or None/'none'")

    xw = x_gate * w

    p = np.zeros_like(x, dtype=float)
    p[idx0:idx1+1] = xw
    return p, (idx0, idx1), dt


#ch = channels["echo0"]
t_s = t
x = amp

p, (i0, i1), dt = build_pulse_record_from_echo0(t_s, x, t0_us=0.0, t1_us=1.0, window="tukey", alpha=0.3)

t_rel = (t_s - t_s[0])

# sanity overlay: p(t) should sit exactly on top of echo0 inside the gate (modulo mean/window)
plt.figure(figsize=(10,4))
plt.plot(t_rel*1e6, x, label="echo0 (measured)", linewidth=1)
plt.plot(t_rel*1e6, p, label="p(t) (gated+windowed, same grid)", linewidth=2)
plt.xlim(0, 5)
plt.xlabel("Time (µs)")
plt.ylabel("Amplitude (arb.)")
plt.title("Pulse record on the original sample grid")
plt.legend()
plt.tight_layout()
plt.show()

print("Gate indices:", i0, i1, " | Gate duration (us):", (t_rel[i1]-t_rel[i0])*1e6)



In [ ]:
# Cell 2 — reconstruct echo-train s(t) from impedance-derived H(f), with matching length

def interp_complex(f_target, f_src, y_src, fill=0.0+0.0j):
    """Linear interpolate complex y_src(f_src) onto f_target."""
    y = np.empty_like(f_target, dtype=np.complex128)
    y[:] = fill
    band = (f_target >= f_src.min()) & (f_target <= f_src.max())
    y.real[band] = np.interp(f_target[band], f_src, np.real(y_src))
    y.imag[band] = np.interp(f_target[band], f_src, np.imag(y_src))
    return y

# --- You must provide these from your impedance calculation/measurement ---
# freqs_Z_Hz : 1D array (Hz)
# Z_complex  : 1D array (Ohms, complex), same length as freqs_Z_Hz

Z0 = 50.0  # ohms reference (change if needed)

N = len(x)
f = np.fft.rfftfreq(N, d=dt)

P = np.fft.rfft(p, n=N)
freqs_Z_Hz=freqs
Gamma_Z = Gamma2       # electrical reflection coefficient
H = interp_complex(f, freqs_Z_Hz, Gamma_Z, fill=0.0)  # put H on the FFT grid

S = H * P
s = np.fft.irfft(S, n=N)  # reconstructed echo-trace, same length as echo0


In [ ]:
# Cell 3 — auto-align (time shift) + scale, then overlay

# Cell — shift-only alignment (no extra scaling)

# Cell — shift-only alignment (no extra scaling)

def best_shift_only(y_ref, y_est):
    N = len(y_ref)
    a = y_ref - np.mean(y_ref)
    b = y_est - np.mean(y_est)
    corr = np.fft.irfft(np.fft.rfft(a) * np.conj(np.fft.rfft(b)), n=N)
    k = int(np.argmax(corr))
    if k > N//2:
        k -= N
    return k



def best_shift_scale(y_ref, y_est, extra_gain=1.0):
    N = len(y_ref)
    a = y_ref - np.mean(y_ref)
    b = y_est - np.mean(y_est)

    corr = np.fft.irfft(np.fft.rfft(a) * np.conj(np.fft.rfft(b)), n=N)
    k = int(np.argmax(corr))
    if k > N//2:
        k -= N

    y_shift = np.roll(y_est, k)
    scale = float(np.dot(y_ref, y_shift) / np.dot(y_shift, y_shift))
    return k, (extra_gain * scale)


# shift_samp, scale = best_shift_scale(x, s)
# s_aligned = scale * np.roll(s, shift_samp)

# print(f"Best shift = {shift_samp} samples = {shift_samp*dt*1e6:.3f} µs,  scale = {scale:.3g}")

# plt.figure(figsize=(10,4))
# plt.plot(t_rel*1e6, x, label="echo0 (measured)", linewidth=1)
# plt.plot(t_rel*1e6, s_aligned, label="reconstructed (aligned+scaled)", linewidth=1.5)
# plt.xlim(0, 20)
# plt.xlabel("Time (µs)")
# plt.ylabel("Amplitude (arb.)")
# plt.title("Echo-train overlay")
# plt.legend()
# plt.tight_layout()
# plt.show()


In [ ]:
pulse_gain=500
k, scale = best_shift_scale(x, s, extra_gain=pulse_gain)
s_aligned = scale * np.roll(s, k)


In [ ]:

# ---------- build reflection spectrum ----------
# Gamma = reflection_coeff(Ze, Z0=50.0)
# Gamma *= window_1d("Hann", len(freqs))
# # ---------- build reflection spectrum ----------
# Gamma2 = reflection_coeff(ZimpE, Z0=50.0)
# Gamma2 *= window_1d("Hann", len(freqs))

# P = gaussian_spectrum_from_time_fwhm(freqs, f0_hz=99.50e6, t_fwhm_s=0.25e-6, t0_s=0.50e-6)
# # ---------- build echo spectrum ----------
# S = Gamma * P*20
# S2 = Gamma2 * P*20
# t2, s_t2 = ifft_one_sided_to_time(f_full2, S_full2, pad_factor=8, make_real=False)


# # IFFT to time
# t, s_t = ifft_one_sided_to_time(f_full, S_full, pad_factor=8, make_real=False)

In [ ]:
# fig, ax = plt.subplots(figsize=(10,4))
# ax.plot(t*1e6, np.real(s_t)*pulse_gain, label="Mason", color="blue")
# ax.plot(t_rel*1e6, x, label="measured echo0", color="orange")
# ax.plot(t2*1e6, np.real(s_t2)*pulse_gain, label="Re(s(t))", color="red")

# ax.set_xlim(1, 20)
# ax.set_ylim(-0.15,0.15)
# ax.set_xlabel("Time (µs)")
# ax.set_ylabel("Amplitude (arb.)")
# ax.legend()
# fig.tight_layout()
# plt.show()



In [ ]:
plt.close('all')